In [1]:
# 0. INITIAL CONFIG

import pandas as pd
import shutil
import os

# Configure so pandas shows all columns when printing Dataframes
pd.set_option('display.max_columns', None)

# Paths for the XAI case base CSV and images directory copies
PATH_XAI_CSV = 'case_base_xai.csv'
PATH_XAI_IMAGES = 'xai_images/'

In [2]:
# 1. COLUMNS TO BE ADAPTED (XAI_GRAPH_TYPE AND ATTRIBUTES)

# Load the XAI case base
df_xai = pd.read_csv(PATH_XAI_CSV)

print(f"Total cases in the XAI case base: {len(df_xai)}")

# Fusion of the library and the graph type, adding a space between them (e.g. "SHAP Summary Plot")
df_xai['graph_type'] = df_xai['library'] + ' ' + df_xai['xai_graph_type']

# Rename 'attributes' column as 'variables'
df_xai.rename(columns={'attributes': 'variables'}, inplace=True)

df_xai

Total cases in the XAI case base: 1382


,id,img_path,domain_emb_path,task_emb_path,visual_emb_path,domain,ai_task,ai_problem_type,class,library,input_format,xai_graph_type,ai_model,explainer,variables,scope,portability,concurrency,analytical_family,solution_insights,qwen_insight,pixtral_insight,idefics_insight,graph_type
0,1,xai_images/000001.png,embeddings_domain/000001.npy,embeddings_task/000001.npy,embeddings_visual/000001.npy,Healthcare,Diabetes progression prediction,Regression,201.42,SHAP,tabular,Waterfall,RandomForestRegressor,TreeExplainer,"age, sex, bmi, bp, s1, s2, s3, s4, s5, s6",Local,Specific,Post,Local_Attribution,The SHAP Waterfall plot for this RandomForestR...,The SHAP Waterfall plot for this local RandomF...,The waterfall graph visualizes the local expla...,The visual graph showcases the impact of vario...,SHAP Waterfall
1,2,xai_images/000002.png,embeddings_domain/000002.npy,embeddings_task/000002.npy,embeddings_visual/000002.npy,Healthcare,Diabetes progression prediction,Regression,201.42,SHAP,tabular,Force,RandomForestRegressor,TreeExplainer,"age, sex, bmi, bp, s1, s2, s3, s4, s5, s6",Local,Specific,Post,Local_Attribution,The local prediction of 201.42 is derived from...,The SHAP Force plot for the local prediction o...,The attached xAI visualization provides a deta...,The visual graph is a force plot that illustra...,SHAP Force
2,3,xai_images/000003.png,embeddings_domain/000003.npy,embeddings_task/000003.npy,embeddings_visual/000003.npy,Healthcare,Diabetes progression prediction,Regression,201.42,SHAP,tabular,Bar,RandomForestRegressor,TreeExplainer,"age, sex, bmi, bp, s1, s2, s3, s4, s5, s6",Local,Specific,Post,Local_Attribution,The xAI visualization analyzes the local impac...,The SHAP bar plot for this local RandomForestR...,The xAI visualization provided illustrates the...,The attached graph is a bar chart that visuall...,SHAP Bar
3,4,xai_images/000004.png,embeddings_domain/000004.npy,embeddings_task/000004.npy,embeddings_visual/000004.npy,Healthcare,Diabetes progression prediction,Regression,NaN,SHAP,tabular,Bar,RandomForestRegressor,TreeExplainer,"age, sex, bmi, bp, s1, s2, s3, s4, s5, s6",Cohort,Specific,Post,Cohort_Pattern,A RandomForestRegressor model predicting diabe...,"The visual bar chart, generated by the SHAP Tr...",The visual graph illustrates the SHAP (SHapley...,The visual graph represents the impact of vari...,SHAP Bar
4,5,xai_images/000005.png,embeddings_domain/000005.npy,embeddings_task/000005.npy,embeddings_visual/000005.npy,Healthcare,Diabetes progression prediction,Regression,NaN,SHAP,tabular,Bar,RandomForestRegressor,TreeExplainer,"age, sex, bmi, bp, s1, s2, s3, s4, s5, s6",Global,Specific,Post,Global_Summary,The visual graph displays mean SHAP values for...,The global SHAP bar plot for the RandomForestR...,"The attached visual graph, generated using SHA...",The visual graph shows the mean SHAP values fo...,SHAP Bar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,1378,xai_images/001378.png,embeddings_domain/001378.npy,embeddings_task/001378.npy,embeddings_visual/001378.npy,Environment,Forest fire burned area prediction,Regression,NaN,ALE,tabular,2D,Keras_DNN,PyALE_Function,"temp, RH",Global,Agnostic,Post,Dependence_Curve,The 2D ALE plot illustrates the effects of rel...,The 2D ALE plot visually demonstrates the glob...,The attached 2D ALE plot provides a visual rep...,The attached graph is a 2D ALE plot that visua...,ALE 2D
1378,1379,xai_images/001379.png,embeddings_domain/001379.npy,embeddings_task/001379.npy,embeddings_visual/001379.npy,Environment,Forest fire burned area prediction,Regression,NaN,ALE,tabular,1D,TorchRegressor,PyALE_Function,temp,Global,Agnostic,Post,Dependence_Curve,In a TorchRegressor forest fire burned area pr...,The 1D ALE plot for the 'temp' feature in the ...,The visual graph provided illustrates the effe...,The attached graph is a 1D ALE plot that illus...,ALE 1D
1379,1380,xai_images/001380.png,embeddings_domain/001380.npy,embeddings_task/0

In [7]:
# 2. SHUFFLE AND DEDUPLICATE THE XAI DATAFRAME TO ENSURE VARIABILITY

# Shuffle the entire DataFrame (frac=1) in a reproducible way (random_state=23)
df_xai_shuffled = df_xai.sample(frac=1, random_state=23).copy()

# Deduplicate using the representative columns: 'ai_task', 'graph_type', 'variables' (the rest are id, paths or insights)
# As we previuosly shuffled, we will keep a random case for these 3 columns (this way we dont always choose the same model,explainer,etc.)
df_uniques = df_xai_shuffled.drop_duplicates(subset=['ai_task', 'graph_type', 'variables']).copy()

print(f"Unique cases after deduplication with columns 'ai_task', 'graph_type', 'variables' for variability: {len(df_uniques)}")

df_uniques

Unique cases after deduplication with columns 'ai_task', 'graph_type', 'variables' for variability: 86


,id,img_path,domain_emb_path,task_emb_path,visual_emb_path,domain,ai_task,ai_problem_type,class,library,input_format,xai_graph_type,ai_model,explainer,variables,scope,portability,concurrency,analytical_family,solution_insights,qwen_insight,pixtral_insight,idefics_insight,graph_type
883,884,xai_images/000884.png,embeddings_domain/000884.npy,embeddings_task/000884.npy,embeddings_visual/000884.npy,Food & Beverage,Wine cultivar classification,Classification,class_1,SHAP,tabular,Bar,TorchMulticlass,GradientExplainer,"alcohol, malic_acid, ash, alcalinity_of_ash, m...",Cohort,Specific,Post,Cohort_Pattern,For the TorchMulticlass model used in wine cul...,"The visual bar chart, generated by the SHAP Gr...",The provided SHAP visualization for the TorchM...,The visual graph represents the impact of vari...,SHAP Bar
772,773,xai_images/000773.png,embeddings_domain/000773.npy,embeddings_task/000773.npy,embeddings_visual/000773.npy,Healthcare,Diabetes progression prediction,Regression,127.78,SHAP,tabular,Bar,TorchRegressor,GradientExplainer,"age, sex, bmi, bp, s1, s2, s3, s4, s5, s6",Local,Specific,Post,Local_Attribution,The SHAP values for the diabetes progression p...,The SHAP bar plot for this local model predict...,"The attached xAI visualization, generated usin...",The visual graph represents the SHAP values fo...,SHAP Bar
575,576,xai_images/000576.png,embeddings_domain/000576.npy,embeddings_task/000576.npy,embeddings_visual/000576.npy,Environment,Forest fire burned area prediction,Regression,NaN,SHAP,tabular,Beeswarm,GradientBoostingRegressor,TreeExplainer,"X, Y, month, day, FFMC, DMC, DC, ISI, temp, RH...",Global,Specific,Post,Global_Summary,"The SHAP beeswarm plot, which utilizes a color...",The SHAP beeswarm plot reveals that among the ...,The visual graph presents a SHAP (SHapley Addi...,The visual graph shows the impact of various f...,SHAP Beeswarm
959,960,xai_images/000960.png,embeddings_domain/000960.npy,embeddings_task/000960.npy,embeddings_visual/000960.npy,Healthcare,Breast cancer classification,Classification,malignant,LIME,tabular,Dashboard,LogisticRegression,LimeTabularExplainer,"mean radius, mean texture, mean perimeter, mea...",Local,Agnostic,Post,Local_Attribution,The model predicts class 1 (malignant) with a ...,The LIME dashboard visually confirms that the ...,"The attached xAI visualization, using LIME for...",The attached graph is a dashboard visualizatio...,LIME Dashboard
451,452,xai_images/000452.png,embeddings_domain/000452.npy,embeddings_task/000452.npy,embeddings_visual/000452.npy,Food & Beverage,Wine cultivar classification,Classification,class_1,SHAP,tabular,Force,KNeighborsClassifier,PermutationExplainer,"alcohol, malic_acid, ash, alcalinity_of_ash, m...",Local,Agnostic,Post,Local_Attribution,"The xAI visualization, generated using SHAP's ...",The Force graph visually confirms that two fea...,"The attached xAI visualization, generated usin...",The visual graph is a horizontal bar chart tha...,SHAP Force
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687,688,xai_images/000688.png,embeddings_domain/000688.npy,embeddings_task/000688.npy,embeddings_visual/000688.npy,Healthcare,Breast cancer classification,Classification,benign,SHAP,tabular,Scatter,RandomForestClassifier,TreeExplainer,worst concave points,Global,Specific,Post,Dependence_Curve,The scatter plot visualizes the relationship b...,The visual evidence in the scatter plot confir...,The attached visual graph provides a detailed ...,The scatter plot visualizes the relationship b...,SHAP Scatter
727,728,xai_images/000728.png,embeddings_domain/000728.npy,embeddings_task/000728.npy,embeddings_visual/000728.npy,Healthcare,Breast cancer classification,Classification,benign,SHAP,tabular,Scatter,LogisticRegression,LinearExplainer,mean area,Global,Specific,Post,Dependence_Curve,This SHAP analysis scatter plot interprets a l...,The visual evidence in the scatter plot confir...,The visualization provided is a sca

In [15]:
# 3. EXTRACTION OF THE 50 CASES TO BE TRASFERED INTO THE NEW CASE BASE
# (we have 12 types of graphs, 5 datasets -> choose 4 cases (datasets) for each graph = 48 cases + 2 extra cases to complete)

# Shuffle the unique cases for variability
df_uniques_shuffled = df_uniques.sample(frac=1, random_state=23)

# Create a strict pool of exactly 60 cases (1 for each graph_type + ai_task combination)
# Dropping duplicates on these two columns guarantees no dataset is repeated within a graph type
df_60_pool = df_uniques_shuffled.drop_duplicates(subset=['graph_type', 'ai_task'])

# Extract exactly 4 cases per graph type (since df_60_pool has unique datasets per graph, selecting 4 guarantees 4 distinct datasets)
df_48 = df_60_pool.groupby('graph_type').head(4)

# Among the 12 cases left behind we randomly pick 2 to complete the 50 cases (ensuring distinct datasets for the graph type)
df_leftover = df_60_pool.drop(df_48.index)
df_extra = df_leftover.sample(n=2, random_state=23)

# Concatenate and shuffle for the final sample of 50 cases
df_50_originals = pd.concat([df_48, df_extra]).sample(frac=1, random_state=23).reset_index(drop=True)

# Save the 50 original cases for future comparison
df_50_originals.to_csv("original_50_xai_cases.csv", index=False)

# EXTRA COMPARISON (DISTRIBUTION OF THE 50 SELECTED CASES)

print("Amount of cases per graph type in the 50 selected cases:")
print(df_50_originals['graph_type'].value_counts(dropna=False))
print("\n")

print("Dataset distribution per graph type in the 50 selected cases:")
# Create a crosstab to see which dataset (ai_task) was assigned to each graph type
distribution_datasets = pd.crosstab(df_50_originals['graph_type'], df_50_originals['ai_task'])
display(distribution_datasets)

Amount of cases per graph type in the 50 selected cases:
graph_type
LIME Bar          5
LIME Dashboard    5
SHAP Violin       4
SHAP Decision     4
SHAP Bar          4
ALE 2D            4
ALE 1D            4
SHAP Waterfall    4
SHAP Scatter      4
SHAP Force        4
SHAP Heatmap      4
SHAP Beeswarm     4
Name: count, dtype: int64


Dataset distribution per graph type in the 50 selected cases:


ai_task,Breast cancer classification,Diabetes progression prediction,Forest fire burned area prediction,Fuel efficiency (MPG) prediction,Wine cultivar classification
graph_type,,,,,
ALE 1D,1,0,1,1,1
ALE 2D,0,1,1,1,1
LIME Bar,1,1,1,1,1
LIME Dashboard,1,1,1,1,1
SHAP Bar,1,0,1,1,1
SHAP Beeswarm,1,1,1,0,1
SHAP Decision,1,1,0,1,1
SHAP Force,1,1,1,0,1
SHAP Heatmap,1,1,1,1,0


In [17]:
# 4. FINAL FORMATTING AND MERGING WITH STATISTICAL CASES

import os
import shutil
import pandas as pd

# Auxiliary function to map the math concept based purely on the XAI library
def get_math_concept(library_name):
    lib = str(library_name).upper()
    if "SHAP" in lib: return "Shapley Values"
    if "LIME" in lib: return "Local Linear Approximation"
    if "ALE" in lib: return "Accumulated Local Effects"
    return "Feature Importance Analysis" # Fallback

current_case_id = 504 
new_rows = []
path_xai_images = 'xai_images/'

os.makedirs("images", exist_ok=True) 

for index, row in df_50_originals.iterrows():

    new_id_str = f"{current_case_id:06d}"
    
    # Copy and rename the image file
    old_filename = os.path.basename(row['img_path']) 
    old_img_path = os.path.join(path_xai_images, old_filename)
    new_img_path = f"images/{new_id_str}.png"
    
    if os.path.exists(old_img_path):
        shutil.copy(old_img_path, new_img_path)
    else:
        print(f"WARNING: Image not found at {old_img_path}")
    
    # New row construction for the CSV
    formatted_row = {
        'id': new_id_str,
        'img_path': new_img_path,
        'domain_emb_path': f"embeddings_domain/{new_id_str}.npy",
        'task_emb_path': f"embeddings_task/{new_id_str}.npy",
        'visual_emb_path': f"embeddings_visual/{new_id_str}.npy",
        'domain': row['domain'],
        'graph_category': "XAI",
        'graph_type': row['graph_type'],
        'math_concept': get_math_concept(row['library']),
        'analytical_task': f"Feature contribution analysis of {row['variables']} via {row['graph_type']}",
        'variables': row['variables'],
        'solution_insights': "",
        'qwen_insight': "",
        'pixtral_insight': "",
        'idefics_insight': ""
    }
    
    new_rows.append(formatted_row)
    current_case_id += 1

# Create the new Dataframe with the formatted XAI cases
df_xai_formatted = pd.DataFrame(new_rows)

# Concatenate with the previous statistical cases and save to a new csv
df_statistics = pd.read_csv("case_base_prev.csv", dtype={'id': str}) # Load the previous 503 cases
df_combined = pd.concat([df_statistics, df_xai_formatted], ignore_index=True) # Combine both datasets ignoring the old indices
df_combined['id'] = df_combined['id'].astype(str).str.zfill(6) # Ensure id format
df_combined.to_csv("case_base_prev_aux.csv", index=False) # Save to the auxiliary CSV file

print(f"Successfully combined! Total cases in the new dataset: {len(df_combined)}")

Successfully combined! Total cases in the new dataset: 553
